# 4.1 Clean publication data

This notebook does the following:
    - Restricts publication data to consenting researchers

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from match_publications_to_labs import match_publications_to_labs

In [2]:
# Load data
labs = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

sensitive_data = pd.read_csv(
    config.SENSITIVE_DATA / "sensitive_data.csv",
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""], # Only treat empty strings as NaN
    usecols=["labgroupid", 
             "responsible_person_surname", 
             "responsible_person_firstname"] # Keep only labgroupid and surname firstname cols
)

publications = pd.read_csv(
    config.PUBLICATON_DATA / "1_Raw" / "zora_metadata.csv",
    usecols=["relation", "creator", "title", "date", "subject", "description"]
)

In [3]:
# Merge the datasets on labgroupid
df_consent = pd.merge(labs, sensitive_data, on="labgroupid", how="left")

# Keep only labgroupids where consent to merge
df_consent = df_consent[df_consent["consent_data_merge"] == "Yes I consent to this data collection and merging"]

# Create df with only labgroupid and researcher name
researchers = df_consent[
    ["labgroupid", "responsible_person_surname", "responsible_person_firstname"]
    ].drop_duplicates()

# Separate researchers into as many cols as needed, based on "," split
researchers["responsible_person_surname"] = researchers["responsible_person_surname"].str.split(",")
researchers["responsible_person_firstname"] = researchers["responsible_person_firstname"].str.split(",")
assert researchers["responsible_person_surname"].apply(len).equals(researchers["responsible_person_firstname"].apply(len)), "Mismatch in number of surnames and firstnames"
max_researchers = researchers["responsible_person_surname"].apply(len).max()
for i in range(max_researchers):
    researchers[f"researcher_surname_{i+1}"] = researchers["responsible_person_surname"].apply(lambda x: x[i] if i < len(x) else None)
    researchers[f"researcher_firstname_{i+1}"] = researchers["responsible_person_firstname"].apply(lambda x: x[i] if i < len(x) else None)
researchers = researchers.drop(columns=["responsible_person_surname", "responsible_person_firstname"])

## (2) Check out publication data

In [4]:
# Print the first 20 rows of creator cols to see format (commented out as sensitive)
# print(publications["creator"].head(20))

## (3) Match publications to labgroupids

In [5]:
# Reshape researchers (currently wide, with paired researcher_surname_i /
# researcher_firstname_i columns) into long format: one row per person,
# keeping each person's surname and firstname together
max_researchers = len([c for c in researchers.columns if c.startswith("researcher_surname_")])

researcher_rows = []
for i in range(1, max_researchers + 1):
    pair = researchers[["labgroupid", f"researcher_surname_{i}", f"researcher_firstname_{i}"]].copy()
    pair.columns = ["labgroupid", "researcher_surname", "researcher_firstname"]
    researcher_rows.append(pair)

researchers_long = pd.concat(researcher_rows, ignore_index=True)
researchers_long = researchers_long.dropna(subset=["researcher_surname", "researcher_firstname"])

In [6]:
# Match each publication's creators against the researchers list
matches = match_publications_to_labs(
    pub_df=publications,
    id_col="relation", 
    creator_col="creator",
    researchers_df=researchers_long,
    surname_col="researcher_surname",
    first_name_col="researcher_firstname",
)

# Report on matching progress
n_total = len(matches)
n_matched = (matches["matched_labgroupids"].str.len() > 0).sum()
print(f"Matched {n_matched} of {n_total} publications to at least one labgroupid.")

/Users/drutna/Library/CloudStorage/OneDrive-UniversitätZürichUZH/lab-experiment/1_Cleaning/match_publications_to_labs.py:469: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  out = creators.groupby(id_col, sort=False).apply(aggregate).reset_index()


Matched 8014 of 207554 publications to at least one labgroupid.


In [7]:
# Check why matches and publications different lengths
x = publications["creator"].isna().sum()  # missing creators
y = publications.shape[0]                 # total publications
z = matches.shape[0]                      # total matches
assert y - z == x # check that missing matches are due to missing creators
print(publications["relation"].duplicated().sum()) # (2) duplicate IDs

0


In [8]:
# Merge matches with publications
publications = publications.merge(matches, on="relation", how="left")

# Keep only rows with at least one matched labgroupid
publications_matched = publications[publications["matched_labgroupids"].notna() &
                                    publications["matched_labgroupids"].str.len() > 0].copy()

In [9]:
# Create column which is length of matched_labgroupids list
publications_matched["n_matched_labgroupids"] = publications_matched["matched_labgroupids"].apply(len)

print(publications_matched["n_matched_labgroupids"].value_counts())

n_matched_labgroupids
1    7609
2     393
3       8
4       4
Name: count, dtype: int64


## Save processed dataset

In [10]:
# Transform list variables to string for saving to CSV
list_cols = ["matched_labgroupids", "match_confidence", "umlaut_variant", "middle_name_match"]
for col in list_cols:
    publications_matched[col] = publications_matched[col].apply(
        lambda x: ";".join(str(v) for v in x) if isinstance(x, list) else x)

In [11]:
# Save processed dataset
publications_matched.to_csv(config.PUBLICATON_DATA / "2_Processed" / "publications_matched.csv", index=False)